### RunnableBranch
RunnableBranch 는 입력에 따라 동적으로 로직을 라우팅할 수 있는 강력한 도구입니다. 이를 통해 개발자는 입력 데이터의 특성에 기반하여 다양한 처리 경로를 유연하게 정의 할 수 있습니다.

RunnableBranch 는 복잡한 의사 결정 트리를 간단하고 직관적인 방식으로 구현할 수 있도록 도와줍니다. 이는 코드의 가독성과 유지보수성을 크게 향상시키며, 로직의 모듈화와 재사용성을 촉진합니다.

또한, RunnableBranch 는 런타임에 동적으로 분기 조건을 평가하고 적절한 처리 루틴을 선택할 수 있어, 시스템의 적응력과 확장성을 높여줍니다.

이러한 특징들로 인해 RunnableBranch는 다양한 도메인에서 활용될 수 있으며, 특히 입력 데이터의 다양성과 변동성이 큰 애플리케이션 개발에 매우 유용합니다. RunnableBranch 를 효과적으로 활용하면 코드의 복잡성을 줄이고, 시스템의 유연성과 성능을 향상시킬 수 있습니다.

### 입력에 따른 동적 로직 라우팅
LangChain Expression Language에서 라우팅을 수행하는 방법에 대해 다룹니다.

라우팅을 통해 이전 단계의 출력이 다음 단계를 정의하는 비결정적 체인을 생성할 수 있습니다. 라우팅은 LLM과의 상호 작용에 구조와 일관성을 제공하는 데 도움이 됩니다.

라우팅을 수행하는 방법에는 두 가지가 있습니다.

RunnableLambda 에서 조건부로 실행 가능한 객체를 반환 (권장)
RunnableBranch 사용
두 가지 방법 모두 첫 번째 단계에서 입력 질문을 수학, 과학 또는 기타에 대한 것으로 분류한 다음, 해당 프롬프트 체인으로 라우팅하는 두 단계 시퀀스를 사용하여 설명하겠습니다.

In [1]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()

True

In [2]:
# LangSmith 추적을 설정합니다. https://smith.langchain.com
# !pip install langchain-teddynote
from langchain_teddynote import logging

# 프로젝트 이름을 입력합니다.
logging.langsmith("CH13-LCEL-Advanced")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH13-LCEL-Advanced



### 간단한 예시
먼저, 들어오는 질문이 수학, 과학, 또는 기타 중 하나로 분류하는 Chain을 생성하겠습니다

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """주어진 사용자 질문을 `수학`, `과학`, 또는 `기타` 중 하나로 분류하세요. 한 단어 이상으로 응답하지 마세요.

<question>
{question}
</question>

Classification:"""
)

# 체인을 생성합니다.
chain = (
    prompt
    | ChatOpenAI(model="gpt-4o-mini")
    | StrOutputParser()  # 문자열 출력 파서를 사용합니다.
)

In [5]:
# 질문을 입력하여 체인을 호출합니다.
chain.invoke({"question": "2+2 는 무엇인가요?"})

'수학'

In [6]:
# 질문을 입력하여 체인을 호출합니다.
chain.invoke({"question": "작용 반작용의 법칙은 무엇인가요?"})

'과학'

In [7]:
# 질문을 입력하여 체인을 호출합니다.
chain.invoke({"question": "Google은 어떤 회사인가요?"})

'기타'

In [8]:
math_chain = (
    PromptTemplate.from_template(
        """You are an expert in math. \
Always answer questions starting with "깨봉선생님께서 말씀하시기를..". \
Respond to the following question:

Question: {question}
Answer:"""
    )
    # OpenAI의 LLM을 사용합니다.
    | ChatOpenAI(model="gpt-4o-mini")
)

science_chain = (
    PromptTemplate.from_template(
        """You are an expert in science. \
Always answer questions starting with "아이작 뉴턴 선생님께서 말씀하시기를..". \
Respond to the following question:

Question: {question}
Answer:"""
    )
    # OpenAI의 LLM을 사용합니다.
    | ChatOpenAI(model="gpt-4o-mini")
)

general_chain = (
    PromptTemplate.from_template(
        """Respond to the following question concisely:

Question: {question}
Answer:"""
    )
    # OpenAI의 LLM을 사용합니다.
    | ChatOpenAI(model="gpt-4o-mini")
)

### 사용자 정의 함수 사용하기
LangChain 공식 도큐먼트에서 권장하는 방식이며, 서로 다른 출력 간의 라우팅을 위해 사용자 정의 함수를 RunnableLambda 로 래핑하여 활용할 수도 있습니다.

In [9]:
def route(info):
    # 주제에 "수학"이 포함되어 있는 경우
    if "수학" in info["topic"].lower():
        # datascience_chain을 반환
        return math_chain
    # 주제에 "과학"이 포함되어 있는 경우
    elif "과학" in info["topic"].lower():
        # art_chain을 반환
        return science_chain
    # 그 외의 경우
    else:
        # general_chain을 반환
        return general_chain

In [10]:
from operator import itemgetter
from langchain_core.runnables import RunnableLambda

full_chain = (
    {"topic": chain, "question": itemgetter("question")}
    | RunnableLambda(
        # 경로를 지정하는 함수를 인자로 전달합니다.
        route
    )
    | StrOutputParser()
)

In [11]:

# 수학과 관련된 질문을 입력하여 체인을 호출합니다.
full_chain.invoke({"question": "미적분의 개념에 대해 말씀해 주세요."})

'깨봉선생님께서 말씀하시기를, 미적분은 수학의 한 분야로서, 함수의 변화율과 면적 문제를 다루는 두 가지 주요 개념인 미분과 적분으로 구성됩니다. 미분은 함수의 변화율, 즉 어떤 함수가 특정 점에서 얼마나 빠르게 변하는지를 측정하는 도구입니다. 이를 통해 그래프의 기울기나 최댓값, 최솟값과 같은 중요한 특성을 분석할 수 있습니다.\n\n반면에 적분은 함수가 나타내는 영역의 크기, 즉 그래프 아래의 면적을 구하는 방법입니다. 적분은 미분과 밀접한 관계가 있으며, 기본 정리에 따르면 미적분은 서로 반대의 과정임을 나타냅니다. 미적분학은 물리학, 공학, 경제학 등 다양한 분야에서 중요한 응용을 가지고 있습니다. 이러한 한계를 넘어서, 미적분은 복잡한 문제를 해결하는 데 강력한 도구가 됩니다.'

In [12]:
# 과학과 관련된 질문을 입력하여 체인을 호출합니다.
full_chain.invoke({"question": "중력은 어떻게 작용하나요?"})

'아이작 뉴턴 선생님께서 말씀하시기를, 중력은 두 물체 사이의 질량에 비례하고 거리의 제곱에 반비례하는 힘으로 작용합니다. 이는 만유인력의 법칙으로 설명되며, 모든 물체는 서로 끌어당기는 힘을 가진다고 하셨습니다. 예를 들어, 지구는 그 크고 큰 질량 덕분에 우리를 끌어당기며, 이로 인해 우리는 지면에 서 있을 수 있습니다. 따라서 중력은 우주에서 물체의 운동을 결정짓는 중요한 힘이라고 할 수 있습니다.'

In [13]:
# 기타 질문을 입력하여 체인을 호출합니다.
full_chain.invoke({"question": "RAG(Retrieval Augmented Generation)은 무엇인가요?"})

'RAG(표 Retrieval Augmented Generation)는 정보를 검색하고 생성하는 기술로, 모델이 외부 데이터베이스나 문서에서 관련 정보를 검색한 후 이를 바탕으로 자연어 응답을 생성하는 방식입니다. 이를 통해 모델의 응답 품질을 향상시키고, 최신 정보를 반영할 수 있습니다.'

### RunnableBranch
RunnableBranch는 입력값에 따라 실행할 조건과 Runnable을 정의할 수 있는 특별한 유형의 Runnable 입니다.

다만, 위에서 설명한 사용자 정의 함수로 구현할 수 없는 기능을 제공하지는 않으므로, 사용자 정의 함수를 사용하는 것이 좋습니다.

문법

- RunnableBranch는 (조건, Runnable) 쌍의 리스트와 기본 Runnable로 초기화됩니다.
- 호출 시 전달된 입력값을 각 조건에 전달하여 분기를 선택합니다.
- True로 평가되는 첫 번째 조건을 선택하고, 해당 조건에 해당하는 Runnable을 입력값과 함께 실행합니다.
- 제공된 조건과 일치하는 것이 없으면 기본 Runnable 을 실행합니다.

In [14]:
from operator import itemgetter
from langchain_core.runnables import RunnableBranch

branch = RunnableBranch(
    # 주제에 "수학"이 포함되어 있는지 확인하고, 포함되어 있다면 math_chain을 실행합니다.
    (lambda x: "수학" in x["topic"].lower(), math_chain),
    # 주제에 "과학"이 포함되어 있는지 확인하고, 포함되어 있다면 science_chain을 실행합니다.
    (lambda x: "과학" in x["topic"].lower(), science_chain),
    # 위의 조건에 해당하지 않는 경우 general_chain을 실행합니다.
    general_chain,
)
# 주제와 질문을 입력받아 branch를 실행하는 전체 체인을 정의합니다.
full_chain = (
    {"topic": chain, "question": itemgetter("question")} | branch | StrOutputParser()
)

In [15]:
# 질문을 입력하여 전체 체인을 실행합니다.
full_chain.invoke({"question": "미적분의 개념에 대해 말씀해 주세요."})

'깨봉선생님께서 말씀하시기를, 미적분은 수학의 한 분야로서, 함수의 변화를 다루는 중요한 도구입니다. 미적분은 크게 두 가지 주요 개념, 즉 미분과 적분으로 나누어집니다.\n\n미분은 함수의 변화율을 계산하는 과정으로, 특정 점에서 함수의 기울기를 찾아내고, 이는 물리적 의미로는 속도와 같은 개념에 연결될 수 있습니다. 예를 들어, 위치에 대한 시간의 함수가 주어졌을 때, 미분을 통해 그 순간의 속도를 알 수 있습니다.\n\n반면에 적분은 함수의 면적을 구하는 과정으로, 특정 구간 동안의 함수의 값을 합산하여 전체적인 양을 구하는 데 사용됩니다. 예를 들어, 속도를 적분하면 이동한 거리를 얻을 수 있습니다.\n\n이 두 개념은 서로 밀접하게 연결되어 있으며, 미적분학의 기본 정리에서는 미분과 적분이 서로의 역과정임을 보여줍니다. 이러한 이유로 미적분은 과학, 공학, 경제학 등 다양한 분야에서 광범위하게 응용되고 있습니다.'

In [16]:
# 질문을 입력하여 전체 체인을 실행합니다.
full_chain.invoke({"question": "중력 가속도는 어떻게 계산하나요?"})

'아이작 뉴턴 선생님께서 말씀하시기를, 중력 가속도는 지구의 중심에서 물체가 받는 중력의 세기를 나타내며, 일반적으로 "g"로 표기합니다. 지구의 중력 가속도는 약 9.81 m/s²로 알려져 있습니다. \n\n중력 가속도를 계산하는 방법은 다음과 같습니다. 만약 특정 높이에서의 중력 가속도를 알고 싶다면, 만유인력 법칙을 이용할 수 있습니다. 만유인력 법칙에 따르면, 두 물체 간의 중력은 그들의 질량의 곱에 비례하고, 거리의 제곱에 반비례합니다. 이를 수식으로 나타내면 다음과 같습니다:\n\n\\[ F = G \\frac{m_1 m_2}{r^2} \\]\n\n여기서 \\( F \\)는 중력, \\( G \\)는 만유인력 상수, \\( m_1 \\)과 \\( m_2 \\)는 두 물체의 질량, \\( r \\)은 두 물체 간의 거리입니다.\n\n이 물리 법칙을 통해 지구의 질량과 반지름을 사용하면 지구 표면에서의 중력 가속도를 계산할 수 있습니다. 이러한 원리를 통해 우리는 중력 가속도의 값을 정확히 이해하고 측정할 수 있습니다.'

In [17]:
# 질문을 입력하여 전체 체인을 실행합니다.
full_chain.invoke({"question": "RAG(Retrieval Augmented Generation)은 무엇인가요?"})

'RAG(정보 검색 강화 생성)는 자연어 처리(NLP) 기술로, 정보 검색과 텍스트 생성을 결합하여 사용자가 질문할 때 외부 데이터베이스에서 관련 정보를 검색한 뒤, 이를 바탕으로 더 풍부하고 정확한 응답을 생성하는 모델입니다. RAG는 대규모 언어 모델과 문서 검색 시스템을 통합하여 성능을 향상합니다.'